# Ingesting an Ontology

`semantic_objects.s223.entities`/`properties`/`enumerationkinds`/`relations` aren't
hand-written - they're generated by parsing the real ASHRAE 223P ontology's SHACL
shapes. This notebook walks through that pipeline: the vendored ontology file, the
parser/IR, the simple-vs-complex shape classification, code generation, and the
generated+hand-written-override split. It's aimed at extending or regenerating the
pipeline, not at everyday library usage - see `s223-generated-classes-tutorial.ipynb`
for that.

## 1. The vendored ontology

Ontology source files are vendored locally (never fetched live at generation time),
under `src/semantic_objects/ontologies/<name>/`, with a manifest recording where they
came from and a checksum for staleness checks.

In [6]:
import json
from pathlib import Path

ontology_dir = Path("../src/semantic_objects/ontologies/s223")
manifest = json.loads((ontology_dir / "MANIFEST.json").read_text())
print(json.dumps(manifest, indent=2))
print()
print((ontology_dir / "223p.ttl").stat().st_size, "bytes")

{
  "source_url": "https://open223.info/223p.ttl",
  "fetched_at": "2026-07-24T00:00:00Z",
  "sha256": "47bdad8925032c84e750e46b3649d102f4e41190c8161df3c9efa3265009b0e0",
  "notes": "Pinned copy of the ASHRAE Standard 223P ontology, vendored for reproducible/offline ontology ingestion. Not fetched live during generation. Re-vendor with scripts/vendor_ontology.py."
}

536776 bytes


## 2. Run the ingestion CLI

Regeneration is explicit and never happens automatically at import/test time.

In [7]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "semantic_objects.ingest.cli", "--ontology", "s223"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr[-500:] if result.stderr else "")

Generated 576 classes and 59 relations into /Users/lazlopaul/Desktop/HPflex/semantic_objects/src/semantic_objects/s223/_generated
22 shape(s) across 8 class(es) could not be resolved into a field/_valid_relations entry (out-of-scope relation namespace or forward bucket reference) - see _generated/_meta.py::UNRESOLVED_NOTES
16 quantity kind(s) referenced by the ontology - see _generated/_meta.py::QUANTITYKINDS_REFERENCED

CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'



## 3. Parse the ontology into an IR, without generating any code

`OntologyParser` walks the ontology graph and produces an ontology-agnostic
intermediate representation (`OntologyIR`): one `ClassIR` per class, one `RelationIR`
per relation. This step alone is useful for inspecting what the ontology actually
contains before committing to generated code.

In [8]:
from semantic_objects.ingest.config import IngestConfig
from semantic_objects.ingest.adapters.s223 import S223Adapter
from semantic_objects.ingest.parser import OntologyParser

config = IngestConfig(
    ontology_name="s223",
    source_path=ontology_dir / "223p.ttl",
    output_dir=Path("/tmp/unused"),  # not writing in this cell, just parsing
)
ir = OntologyParser(config, S223Adapter()).parse()
print(f"{len(ir.classes)} classes, {len(ir.relations)} relations, "
      f"{len(ir.quantitykinds_referenced)} quantity kinds referenced")

CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'


576 classes, 59 relations, 16 quantity kinds referenced


### The `S223Adapter`: ontology-specific categorization

The parser itself is ontology-agnostic - it never references s223-specific IRIs
directly. All of that lives in `S223Adapter` (`ingest/adapters/s223.py`), which
answers four questions: is this subject a class? a relation? abstract? which
generated module does it belong in? A future Brick or WATR adapter implements the
same four methods against a different meta-vocabulary; the parser, SHACL classifier,
and emitter are untouched.

In [11]:
pump = ir.classes["Pump"]
print("Pump ontology IRI:", pump.iri)
print("bucket:", pump.bucket, " abstract:", pump.is_abstract)
print("parent(s):", pump.parent_local_names)

Pump ontology IRI: http://data.ashrae.org/standard223#Pump
bucket: entities  abstract: False
parent(s): ['Equipment']


## 4. Simple vs. qualified vs. complex SHACL shapes

Not every SHACL `sh:property` shape becomes a Python field. The classifier
(`ingest/shacl.py`) sorts them into three outcomes:

- **Plain field**: a single `sh:path` with a direct `sh:class`/`sh:datatype` -> becomes
  an ordinary `required_field()`.
- **Qualified field**: a `sh:path` with an `sh:qualifiedValueShape` anchored by
  `sh:class` -> becomes a `required_field(qualified=True)`, *even if* the qualified
  shape has extra nested constraints beyond the class (those are preserved as a
  supplementary note on the field, not used to block generation).
- **Complex / non-field**: `sh:sparql` constraints, bare `sh:or`, forbidden
  (`maxCount=0`) properties, or compound/inverse paths - none of these anchor to a
  single class, so no field is generated. They're preserved in
  `_generated/_raw_shapes.py` instead of silently dropped.

`Pump` has one of each, all sharing the `hasConnectionPoint` relation:

In [12]:
for shape in pump.property_shapes:
    print(f"FIELD  {shape.field_name:26s} qualified={shape.qualified!s:5} "
          f"target={shape.target_class_local:24s} notes={len(shape.supplementary_notes)}")
for cc in pump.complex_constraints:
    print(f"COMPLEX (standalone) kind={cc.kind} path={cc.path_local}")
for shape in pump.property_shapes:
    for note in shape.supplementary_notes:
        print(f"  note on {shape.field_name}: [{note.kind}] {note.message or note.comment}")

FIELD  outlet_connection_point    qualified=True  target=OutletConnectionPoint    notes=2
FIELD  inlet_connection_point     qualified=True  target=InletConnectionPoint     notes=2
  note on outlet_connection_point: [nested-node] s223: A `Pump` shall have at least one outlet using the medium `Fluid-Water`, `Fluid-Oil` or `Constituent-Refrigerant`.
  note on outlet_connection_point: [sparql] The non-electrical `ConnectionPoint`s of a `Pump` must have compatible media.
  note on inlet_connection_point: [nested-node] s223: A `Pump` shall have at least one inlet using the medium `Fluid-Water`, `Fluid-Oil` or `Constituent-Refrigerant`.
  note on inlet_connection_point: [sparql] The non-electrical `ConnectionPoint`s of a `Pump` must have compatible media.


## 5. What the emitter produces

`Emitter` (`ingest/codegen/emitter.py`) turns the IR into deterministic, header-stamped
`.py` source - sorted by IRI, topologically ordered so a class's dependencies (parents,
same-bucket field types) are always defined earlier in the file, with cross-bucket
references (`enumerationkinds` -> `properties` -> `entities`) resolved via module
prefixing. Self-referential fields (a class relating to its own type, like
`PhysicalSpace.contains`) use `typing.Self`, resolved at use-time by the framework's
existing `_infer_relation_for_field`, rather than needing the class to already exist
at definition time.

In [14]:
from semantic_objects.s223._generated import entities as gen_entities

print(gen_entities.Pump.__dataclass_fields__.keys())
print()
print("PhysicalSpace._valid_relations (note the typing.Self for the self-referential "
      "'contains' relation):")
print(gen_entities.PhysicalSpace._valid_relations)

dict_keys(['outlet_connection_point', 'inlet_connection_point'])

PhysicalSpace._valid_relations (note the typing.Self for the self-referential 'contains' relation):
[(<class 'semantic_objects.s223._generated.relations.contains'>, typing.Self), (<class 'semantic_objects.s223._generated.relations.encloses'>, <class 'semantic_objects.s223._generated.entities.DomainSpace'>)]


## 6. Generated base + hand-written override

Regeneration only ever writes inside `s223/_generated/` - it never touches the
sibling hand-written files (`s223/entities.py`, `s223/properties.py`, ...), which
just `from ._generated.X import *` plus add real customization that can't be derived
from SHACL. `s223/properties.py` is the clearest example: the ontology's
`QuantifiableProperty` shapes reference `qudt:hasQuantityKind`/`qudt:hasUnit`, which
live outside the `s223:` namespace this pilot ingests, so the generated
`QuantifiableObservableProperty` has no `qk`/`value`/`unit` fields at all - those are
added by hand on a subclass.

In [16]:
import inspect
from semantic_objects.s223 import properties

print(inspect.getsource(properties.QuantifiableObservableProperty))

@semantic_object
class QuantifiableObservableProperty(_GeneratedQOP):
    qk: quantitykinds.QuantityKind = required_field(relation=hasQuantityKind, qualified=False)
    value: float = required_field(relation=hasValue)
    unit: Optional[Unit] = field(default=None, metadata={'relation': hasUnit, 'min': 1, 'max': None, 'qualified': True})

    def __post_init__(self):
        """Set default unit if not provided"""
        super().__post_init__()
        if self.unit is None:
            self.unit = DEFAULT_UNIT_MAP[self.qk][DEFAULT_UNIT_SYSTEM]



## 7. Extending to another ontology

Adding Brick or WATR support means writing a new `OntologyAdapter` (namespace scope,
`is_class`/`is_relation`/`is_abstract`/`bucket_for`/`scaffold_parent_local_names`) and
vendoring that ontology's `.ttl` under `ontologies/<name>/` - the parser, SHACL
classifier, and emitter are all already ontology-agnostic. See
`src/semantic_objects/ingest/adapters/base.py` for the interface, and
`adapters/s223.py` for a worked example.